In [2]:
# 📚 Essential Imports and Setup

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
import warnings
warnings.filterwarnings('ignore')

# 📁 Snapshot paths
SNAP_PATH = Path("C:/Users/Marco.Africani/OneDrive - AVU SA/AVU CPI Campaign/Puzzle_control_Reports/IRON_DATA/snapshots")

print("📚 Imports loaded successfully")
print(f"📁 Snapshot path: {SNAP_PATH}")
print(f"📊 Ready for analysis!")

📚 Imports loaded successfully
📁 Snapshot path: C:\Users\Marco.Africani\OneDrive - AVU SA\AVU CPI Campaign\Puzzle_control_Reports\IRON_DATA\snapshots
📊 Ready for analysis!


In [3]:
# 🧊 SNAPSHOT DATA LOADING - Reliable

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path

print("🧊 SNAPSHOT DATA LOADING")
print("=" * 60)

# Setup paths and runtime
SNAP_PATH = Path(r"C:\Users\Marco.Africani\OneDrive - AVU SA\AVU CPI Campaign\Puzzle_control_Reports\IRON_DATA\snapshots")
analysis_runtime = datetime.now()

print(f"📅 Analysis Runtime: {analysis_runtime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📁 Snapshot Directory: {SNAP_PATH}")

# Time windows for SPEC compliance
time_windows = {
    '7d': analysis_runtime - timedelta(days=7),
    '14d': analysis_runtime - timedelta(days=14),
    '30d': analysis_runtime - timedelta(days=30)
}

print(f"\n⏰ Analysis Time Windows:")
for window, cutoff in time_windows.items():
    print(f"   {window}: Campaigns after {cutoff.strftime('%Y-%m-%d %H:%M')}")

# Load snapshot files
snapshot_files = {
    'omt_main_offer.pkl': 'OMT Main Offer',
    'powerbi_stats.pkl': 'Power BI Statistics', 
    'detailed_stock_list.pkl': 'Stock List',
    'clients_lines.pkl': 'Client Lines',
    'inactive2025.pkl': 'Inactive Clients'
}

loaded_data = {}
print(f"\n📊 Loading Snapshot Files:")

for filename, description in snapshot_files.items():
    try:
        snapshot_file = SNAP_PATH / filename
        if snapshot_file.exists():
            df = pd.read_pickle(snapshot_file)
            loaded_data[description] = df
            print(f"   ✅ {description}: {len(df):,} records × {len(df.columns)} columns")
        else:
            print(f"   ❌ {description}: Snapshot file not found")
            loaded_data[description] = None
    except Exception as e:
        print(f"   ❌ {description}: Error loading - {str(e)[:50]}...")
        loaded_data[description] = None

# Assign to global variables with proper names
omt_df = loaded_data.get('OMT Main Offer')
stats_df = loaded_data.get('Power BI Statistics')
stock_df = loaded_data.get('Stock List')
lines_df = loaded_data.get('Client Lines')
inactive_df = loaded_data.get('Inactive Clients')

# Quick data validation and column identification
print(f"\n🔍 Data Validation & Column Mapping:")

# OMT Main Offer validation
if omt_df is not None:
    print(f"   📋 OMT Main Offer ({len(omt_df)} campaigns):")
    schedule_cols = [col for col in omt_df.columns if 'schedule' in col.lower()]
    campaign_cols = [col for col in omt_df.columns if 'campaign' in col.lower()]
    email_cols = [col for col in omt_df.columns if 'email' in col.lower() or 'sent' in col.lower()]
    
    if schedule_cols:
        schedule_col = schedule_cols[0]
        print(f"      📅 Schedule column: '{schedule_col}'")
        try:
            omt_df['schedule_datetime'] = pd.to_datetime(omt_df[schedule_col], errors='coerce')
            valid_schedules = omt_df['schedule_datetime'].notna().sum()
            print(f"      ✅ Valid schedule dates: {valid_schedules}")
        except:
            print(f"      ⚠️  Schedule date parsing failed")
    
    if campaign_cols:
        campaign_col_omt = campaign_cols[0]
        print(f"      📋 Campaign column: '{campaign_col_omt}'")
    
    if email_cols:
        email_col = email_cols[0]
        print(f"      📧 Email column: '{email_col}'")

# Power BI Statistics validation (SPEC compliance)
if stats_df is not None:
    print(f"   📊 Power BI Statistics ({len(stats_df)} records):")
    
    # Find SPEC-required columns
    campaign_cols = [col for col in stats_df.columns if 'campaign' in col.lower() and 'no' in col.lower() and 'main' not in col.lower()]
    main_campaign_cols = [col for col in stats_df.columns if 'main' in col.lower() and 'campaign' in col.lower()]
    lcy_cols = [col for col in stats_df.columns if 'bottle' in col.lower() and 'lcy' in col.lower()]
    customer_cols = [col for col in stats_df.columns if 'customer' in col.lower()]
    segment_cols = [col for col in stats_df.columns if 'segment' in col.lower()]
    wine_cols = [col for col in stats_df.columns if 'wine' in col.lower()]
    size_cols = [col for col in stats_df.columns if 'bottle' in col.lower() and 'size' in col.lower()]
    
    # Store key columns globally
    campaign_col = campaign_cols[0] if campaign_cols else None
    main_campaign_col = main_campaign_cols[0] if main_campaign_cols else None
    lcy_col = lcy_cols[0] if lcy_cols else None
    customer_col = customer_cols[0] if customer_cols else None
    segment_col = segment_cols[0] if segment_cols else None
    wine_col = wine_cols[0] if wine_cols else None
    size_col = size_cols[0] if size_cols else None
    
    print(f"      📋 Campaign No: '{campaign_col}'")
    print(f"      📋 Main Campaign: '{main_campaign_col}'")
    print(f"      💰 LCY Amount: '{lcy_col}' {'✅ SPEC COMPLIANT' if lcy_col else '❌ NOT FOUND'}")
    print(f"      👤 Customer No: '{customer_col}'")
    print(f"      🏢 Segment: '{segment_col}'")
    print(f"      🍷 Wine Name: '{wine_col}'")
    print(f"      📏 Size: '{size_col}'")
    
    # SPEC: HORECA and Trade exclusion
    if segment_col:
        initial_count = len(stats_df)
        exclude_mask = stats_df[segment_col].str.contains('HORECA|Trade|Horeca|trade', case=False, na=False)
        stats_df = stats_df[~exclude_mask]
        excluded_count = initial_count - len(stats_df)
        print(f"      🚫 HORECA & Trade excluded: {excluded_count:,} records (remaining: {len(stats_df):,})")

# Client Lines validation
if lines_df is not None:
    print(f"   👥 Client Lines ({len(lines_df)} clients):")
    if len(lines_df.columns) >= 21:
        contact_col = lines_df.columns[20]  # Column U (index 20)
        print(f"      📞 Contact No (Col U): '{contact_col}'")
        unique_contacts = lines_df[contact_col].nunique()
        print(f"      👤 Unique contacts: {unique_contacts}")
    else:
        print(f"      ⚠️  Column U not available (only {len(lines_df.columns)} columns)")
    
    # Find spending column
    spending_cols = [col for col in lines_df.columns if 'spend' in col.lower() or 'avg' in col.lower()]
    if spending_cols:
        spending_col = spending_cols[0]
        print(f"      💰 Spending column: '{spending_col}'")

# Inactive clients validation
if inactive_df is not None:
    print(f"   😴 Inactive Clients ({len(inactive_df)} records):")
    contact_cols = [col for col in inactive_df.columns if 'contact' in col.lower()]
    if contact_cols:
        inactive_contact_col = contact_cols[0]
        print(f"      📞 Contact column: '{inactive_contact_col}'")

# Stock validation
if stock_df is not None:
    print(f"   📦 Stock List ({len(stock_df)} items):")
    item_cols = [col for col in stock_df.columns if 'item' in col.lower()]
    stock_cols = [col for col in stock_df.columns if 'stock' in col.lower() or 'qty' in col.lower()]
    if item_cols and stock_cols:
        item_col = item_cols[0]
        stock_col = stock_cols[0]
        print(f"      📋 Item column: '{item_col}'")
        print(f"      📊 Stock column: '{stock_col}'")

# Time window filtering for OMT campaigns (SPEC compliance)
if omt_df is not None and 'schedule_datetime' in omt_df.columns:
    print(f"\n⏰ OMT Campaign Filtering by Schedule DateTime:")
    omt_filtered = {}
    
    for window, cutoff_date in time_windows.items():
        window_campaigns = omt_df[omt_df['schedule_datetime'] >= cutoff_date]
        omt_filtered[window] = window_campaigns
        print(f"   {window}: {len(window_campaigns)} campaigns scheduled after {cutoff_date.strftime('%Y-%m-%d')}")
        
        # Show sample campaigns
        if len(window_campaigns) > 0:
            sample = window_campaigns.head(3)
            for idx, row in sample.iterrows():
                campaign_id = row.get(campaign_col_omt if 'campaign_col_omt' in locals() else 'Campaign No.', 'Unknown')
                schedule_date = row['schedule_datetime']
                print(f"      • {campaign_id}: {schedule_date.strftime('%Y-%m-%d %H:%M') if pd.notna(schedule_date) else 'Invalid date'}")
else:
    print(f"\n⚠️  Cannot perform time filtering - OMT schedule data unavailable")
    omt_filtered = {}

# Summary
print(f"\n" + "=" * 60)
print(f"📋 SNAPSHOT LOADING SUMMARY")
print(f"=" * 60)

loaded_count = sum(1 for df in loaded_data.values() if df is not None)
total_records = sum(len(df) for df in loaded_data.values() if df is not None)

print(f"✅ Files loaded: {loaded_count}/{len(snapshot_files)}")
print(f"📊 Total records: {total_records:,}")

for description, df in loaded_data.items():
    status = "✅ Ready" if df is not None else "❌ Missing"
    print(f"{status} {description}")

if omt_filtered:
    total_campaigns_in_windows = sum(len(df) for df in omt_filtered.values())
    print(f"\n🎯 Campaigns in analysis windows: {total_campaigns_in_windows}")

print(f"\n🚀 READY FOR SPEC-COMPLIANT BUNDLE ANALYSIS")
print(f"📅 Loaded at: {analysis_runtime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"=" * 60)

🧊 SNAPSHOT DATA LOADING
📅 Analysis Runtime: 2025-09-30 08:41:13
📁 Snapshot Directory: C:\Users\Marco.Africani\OneDrive - AVU SA\AVU CPI Campaign\Puzzle_control_Reports\IRON_DATA\snapshots

⏰ Analysis Time Windows:
   7d: Campaigns after 2025-09-23 08:41
   14d: Campaigns after 2025-09-16 08:41
   30d: Campaigns after 2025-08-31 08:41

📊 Loading Snapshot Files:
   ✅ OMT Main Offer: 2,419 records × 21 columns
   ✅ Power BI Statistics: 30,516 records × 58 columns
   ✅ Stock List: 4,715 records × 42 columns
   ✅ Client Lines: 61 records × 30 columns
   ✅ Inactive Clients: 2,429 records × 29 columns

🔍 Data Validation & Column Mapping:
   📋 OMT Main Offer (2419 campaigns):
      📅 Schedule column: 'schedule datetime'
      ✅ Valid schedule dates: 2419
      📋 Campaign column: 'campaign status'
      📧 Email column: 'number of sent emails'
   📊 Power BI Statistics (30516 records):
      📋 Campaign No: 'campaign no.'
      📋 Main Campaign: 'main campaign no.'
      💰 LCY Amount: 'total bottle

In [7]:
# === SPEC-COMPLIANT WINNERS (ENHANCED: MinQty=0, Multi-source Vintage) ===
import re
import numpy as np
import pandas as pd

CONV_WEIGHT, SALES_WEIGHT = 0.6, 0.4
RESURRECTION_BONUS_PER = 0.02
RESURRECTION_CAP = 0.10

def _norm_cm(s):
    if pd.isna(s): return ""
    s = str(s).strip()
    s = re.sub(r"\.0$", "", s)
    s = re.sub(r"[^A-Za-z0-9]", "", s).upper()
    s = re.sub(r"^CM", "", s)
    return s.lstrip("0")

def _norm_item(x): 
    return str(x).strip().replace(".0","")

def _looks_75cl(val: str) -> bool:
    return bool(re.search(r"(75\s*cl|0\.75\s*l|750\s*ml|0,75\s*l)", str(val).lower()))

def _minmax(x: pd.Series) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce").fillna(0.0)
    if x.empty: return x
    lo, hi = float(x.min()), float(x.max())
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.ones(len(x)), index=x.index)
    return (x - lo) / (hi - lo)

# ---- Column picks (tolerant) ----
def _pick(cols, *names, regex=None):
    cols = [str(c).lower() for c in cols]
    # exact-ish first
    for n in names:
        if n and n.lower() in cols:
            i = cols.index(n.lower()); return i
    # regex fallback
    if regex:
        rx = re.compile(regex, re.I)
        for i, c in enumerate(cols):
            if rx.search(c): return i
    return None

# === 1) Build bundle_id on stats (HORECA excluded) ===
df = stats_df.copy()

# Identify columns
i_cm   = _pick(df.columns, "campaign no.", "campaign code", regex=r"\bcampaign\b.*(no|code)")
i_main = _pick(df.columns, "main campaign no.", "parent campaign no.", regex=r"^main.*campaign")
i_amt  = _pick(df.columns, "total bottle amount (lcy)", regex=r"total.*bottle.*amount.*lcy")
i_qty  = _pick(df.columns, "total bottle quantity", "quantity", "qty", regex=r"(bottle.*)?qty|quantity")
i_cust = _pick(df.columns, "customer no.", "contact no.", regex=r"^(customer|contact).*(no|id)")
i_item = _pick(df.columns, "item no.", "item number", regex=r"^item(\s|_)?(no|number)?$")
i_size = _pick(df.columns, "bottle size", "item size", "volume", regex=r"(bottle|item|unit|uom).*(size|vol|ml|cl|l)")
i_date = _pick(df.columns, "posting date", "document date", "sales date", "date", regex=r"(posting|document|sales).*date")
i_seg  = _pick(df.columns, "segment code", "segment", regex=r"segment.*code")
i_name = _pick(df.columns, "wine name", "item description", "description", "name", regex=r"(wine\s*name|item\s*description|description|name)")

if i_seg is not None:
    mask_exclude = df.iloc[:, i_seg].astype(str).str.strip().str.lower().isin(["horeca", "trade"])
    df = df.loc[~mask_exclude].copy()

df["cm_no"]     = df.iloc[:, i_cm].map(_norm_cm) if i_cm is not None else ""
df["main_cm"]   = df.iloc[:, i_main].map(_norm_cm) if i_main is not None else ""
df["bundle_id"] = np.where(df["main_cm"]!="", df["main_cm"], df["cm_no"])

# Bundle ID calculation (silent)
campaigns_with_main = (df["main_cm"] != "").sum()
campaigns_standalone = (df["main_cm"] == "").sum()

if i_amt is not None:
    df["amount"] = pd.to_numeric(df.iloc[:, i_amt], errors="coerce").fillna(0.0)
else:
    df["amount"] = 0.0

if i_qty  is not None: df["qty"]       = pd.to_numeric(df.iloc[:, i_qty], errors="coerce")
if i_cust is not None: df["customer_no"] = df.iloc[:, i_cust].astype(str).str.strip().str.replace(r"\.0$","", regex=True)
if i_item is not None: df["item_no"]     = df.iloc[:, i_item].astype(str).str.strip().str.replace(r"\.0$","", regex=True)
if i_size is not None: df["size_raw"]    = df.iloc[:, i_size].astype(str).str.strip().str.lower()
if i_date is not None: df["line_date"]   = pd.to_datetime(df.iloc[:, i_date], errors="coerce")
name_col = df.columns[i_name] if i_name is not None else None

# === 2) CM→bundle map (for proper OMT aggregation & Lines mapping) ===
# Include phantom campaigns (exist in Main Campaign but not in Campaign No.)

# Get all campaigns mentioned in Main Campaign column
all_main_campaigns = set(df[df["main_cm"] != ""]["main_cm"].unique())

# Get all actual campaign numbers
all_actual_campaigns = set(df["cm_no"].unique()) - {""}

# Find phantom campaigns (referenced as main but don't exist as actual campaigns)
phantom_campaigns = all_main_campaigns - all_actual_campaigns

# Create base CM→bundle mapping from actual campaigns
cm_to_bundle = df[["cm_no","bundle_id"]].drop_duplicates()

# Add phantom campaigns to the mapping (they bundle to themselves)
if phantom_campaigns:
    phantom_df = pd.DataFrame({
        "cm_no": list(phantom_campaigns),
        "bundle_id": list(phantom_campaigns)
    })
    cm_to_bundle = pd.concat([cm_to_bundle, phantom_df], ignore_index=True)

# === 3) OMT emails per bundle (Size=75.00, MinQty=0) ===
omt = omt_df.copy()

# Debug: Check OMT columns
print(f"\n🔍 OMT COLUMN DEBUGGING:")
print(f"   • Total columns: {len(omt.columns)}")
print(f"   • All columns: {list(omt.columns)}")

# Look for campaign columns with more flexible regex
campaign_candidates = []
for i, col in enumerate(omt.columns):
    if any(word in str(col).lower() for word in ['campaign', 'cm', 'no.', 'id']):
        campaign_candidates.append((i, col))

email_candidates = []
for i, col in enumerate(omt.columns):
    if any(word in str(col).lower() for word in ['email', 'sent', 'mail']):
        email_candidates.append((i, col))

print(f"   • Campaign column candidates: {campaign_candidates}")
print(f"   • Email column candidates: {email_candidates}")

j_cm = _pick(omt.columns, "campaign no.", "campaign code", "campaign number", regex=r"(campaign).*(no|code|number)")
j_em = _pick(omt.columns, "number of sent emails", "sent emails", "emails sent",
             regex=r"^number.*sent.*email|^sent.*email|emails")

# If not found, try broader search with more specific patterns
if j_cm is None:
    print(f"   • Standard campaign column not found, trying alternatives...")
    # Look for columns that specifically contain campaign numbers (not status)
    for i, col in enumerate(omt.columns):
        col_lower = str(col).lower()
        # Skip campaign status, look for actual campaign number columns
        if ('campaign' in col_lower and any(word in col_lower for word in ['no', 'number', 'id']) 
            and 'status' not in col_lower):
            j_cm = i
            print(f"   • Using campaign column {i}: '{col}'")
            break
    
    # If still not found, look for any 'no.' column that might be campaign number
    if j_cm is None:
        for i, col in enumerate(omt.columns):
            col_lower = str(col).lower()
            if 'no.' in col_lower and 'campaign' not in col_lower:
                j_cm = i
                print(f"   • Using number column {i}: '{col}'")
                break

if j_em is None:
    print(f"   • Standard email column not found, trying alternatives...")
    # Try finding any column with 'email' or 'sent'
    for i, col in enumerate(omt.columns):
        if any(word in str(col).lower() for word in ['email', 'sent', 'mail']):
            j_em = i
            print(f"   • Using email column {i}: '{col}'")
            break

j_size = _pick(omt.columns, "size", "bottle size", "volume", regex=r"^(size|bottle.*size|volume)$")
j_minqty = _pick(omt.columns, "minimum quantity", "min quantity", "min qty", regex=r"(minimum|min).*(quantity|qty)")
j_vintage = _pick(omt.columns, "vintage", regex=r"^vintage$")  # Column N

print(f"   • Selected indices - Campaign: {j_cm}, Email: {j_em}, Size: {j_size}, MinQty: {j_minqty}, Vintage: {j_vintage}")

if j_cm is not None:
    print(f"   • Campaign column: '{omt.columns[j_cm]}'")
if j_em is not None:
    print(f"   • Email column: '{omt.columns[j_em]}'")

assert j_cm is not None and j_em is not None, "OMT must have Campaign No. and Number of Sent Emails."

# Build OMT data with all filtering columns
omt_cols = [j_cm, j_em]
col_names = ["cm_no", "emails_sent"]

if j_size is not None:
    omt_cols.append(j_size)
    col_names.append("size")
if j_minqty is not None:
    omt_cols.append(j_minqty)
    col_names.append("min_qty")
if j_vintage is not None:
    omt_cols.append(j_vintage)
    col_names.append("vintage")

omt_use = omt.iloc[:, omt_cols].copy()
omt_use.columns = col_names

# Apply filters step by step
filter_steps = []
original_count = len(omt_use)

if "size" in omt_use.columns:
    omt_use["size"] = pd.to_numeric(omt_use["size"], errors="coerce")
    size_before = len(omt_use)
    omt_use = omt_use[omt_use["size"] == 75.00]
    filter_steps.append(f"Size=75.00: {size_before}→{len(omt_use)}")

if "min_qty" in omt_use.columns:
    omt_use["min_qty"] = pd.to_numeric(omt_use["min_qty"], errors="coerce")
    minqty_before = len(omt_use)
    omt_use = omt_use[omt_use["min_qty"] == 0]
    filter_steps.append(f"MinQty=0: {minqty_before}→{len(omt_use)}")

# Create vintage mapping from OMT
omt_vintage_map = pd.DataFrame(columns=["cm_no", "omt_vintage"])
if "vintage" in omt_use.columns:
    omt_vintage_map = omt_use[["cm_no", "vintage"]].copy()
    omt_vintage_map.columns = ["cm_no", "omt_vintage"]
    omt_vintage_map["cm_no"] = omt_vintage_map["cm_no"].map(_norm_cm)
    omt_vintage_map["omt_vintage"] = omt_vintage_map["omt_vintage"].astype(str).str.strip()
    omt_vintage_map = omt_vintage_map[~omt_vintage_map["omt_vintage"].isin(["", "nan", "None", "0", "NaN"])]

# Process email data
omt_use["cm_no"] = omt_use["cm_no"].map(_norm_cm)
omt_use["emails_sent"] = pd.to_numeric(omt_use["emails_sent"], errors="coerce").fillna(0).astype(int)

# Debug: Search for our target campaigns in OMT
print(f"\n🔍 SEARCHING FOR TARGET CAMPAIGNS IN OMT:")
target_searches = ["2502493", "2502472", "25-02493", "25-02472"]
for target in target_searches:
    matches = omt_use[omt_use["cm_no"].str.contains(target, na=False)]
    print(f"   • Searching for '{target}': {len(matches)} matches")
    if len(matches) > 0:
        for idx, row in matches.head(3).iterrows():
            print(f"     - CM: '{row['cm_no']}', Emails: {row['emails_sent']:,}")

# Also check original data before normalization
print(f"\n🔍 CHECKING ORIGINAL OMT DATA (before normalization):")
original_cm_col = omt.columns[j_cm]
for target in ["CM-25-02493", "CM-25-02472", "2502493", "2502472"]:
    matches = omt[omt[original_cm_col].astype(str).str.contains(target, na=False)]
    print(f"   • Original search for '{target}': {len(matches)} matches")
    if len(matches) > 0:
        for idx, row in matches.head(3).iterrows():
            print(f"     - Original: '{row[original_cm_col]}', Emails: {row[omt.columns[j_em]]}")

# Handle duplicates (keep first occurrence - silent processing)
duplicate_campaigns = omt_use.duplicated(subset=["cm_no"], keep=False)
num_duplicates = duplicate_campaigns.sum()
omt_unique = omt_use.drop_duplicates(subset=["cm_no"], keep="first")

# Enhanced email aggregation including phantom campaigns (silent processing)
# Map OMT emails to bundles (includes both actual and phantom campaigns)
emails_map = omt_unique.merge(cm_to_bundle, on="cm_no", how="inner")

# Check for phantom campaign emails in OMT
phantom_in_omt = emails_map[emails_map["cm_no"].isin(phantom_campaigns)]
phantom_email_total = phantom_in_omt["emails_sent"].sum() if len(phantom_in_omt) > 0 else 0

# Aggregate emails by bundle (includes phantom campaign emails)
emails_by_bundle = (emails_map.groupby("bundle_id")["emails_sent"].sum()
                    .rename("emails_total").reset_index())

total_emails_all = emails_by_bundle["emails_total"].sum()

# === 4) Inactive recipients per bundle (Lines ∩ inactive2025) ===
inactive_count = pd.DataFrame(columns=["bundle_id","inactive_recipients"])
if lines_df is not None and not lines_df.empty and inactive_df is not None and not inactive_df.empty:
    # Pick columns from Lines: Campaign + Contact
    L_cm = _pick(lines_df.columns, "campaign no.", "campaign code", "campaign id", regex=r"\bcampaign\b")
    L_ct = _pick(lines_df.columns, "contact no.", "customer no.", "contact id", regex=r"^(contact|customer).*(no|id)")
    I_ct = _pick(inactive_df.columns, "contact no.", "customer no.", "contact id", regex=r"^(contact|customer).*(no|id)")
    I_cm = _pick(inactive_df.columns, "campaign no.", "campaign code", "campaign id", regex=r"\bcampaign\b")

    if L_cm is not None and L_ct is not None and I_ct is not None:
        L = lines_df.iloc[:, [L_cm, L_ct]].copy()
        L.columns = ["cm_no","contact_no"]
        L["cm_no"] = L["cm_no"].map(_norm_cm)
        L["contact_no"] = L["contact_no"].astype(str).str.strip().str.replace(r"\.0$","", regex=True)

        if I_cm is not None:
            I = inactive_df.iloc[:, [I_cm, I_ct]].copy()
            I.columns = ["cm_no","contact_no"]
            I["cm_no"] = I["cm_no"].map(_norm_cm)
            I["contact_no"] = I["contact_no"].astype(str).str.strip().str.replace(r"\.0$","", regex=True)
            # Enhanced: Include phantom campaigns in inactive calculation
            LI = (L.merge(cm_to_bundle, on="cm_no", how="inner")
                    .merge(I, on=["cm_no","contact_no"], how="inner"))
        else:
            I = inactive_df.iloc[:, [I_ct]].copy()
            I.columns = ["contact_no"]
            I["contact_no"] = I["contact_no"].astype(str).str.strip().str.replace(r"\.0$","", regex=True)
            # Enhanced: Include phantom campaigns in inactive calculation
            LI = (L.merge(cm_to_bundle, on="cm_no", how="inner")
                    .merge(I, on="contact_no", how="inner"))

        inactive_count = (LI.groupby("bundle_id")["contact_no"].nunique()
                            .rename("inactive_recipients").reset_index())

# === 5) Enhanced Core bundle metrics (includes phantom campaign sales) ===
# CRITICAL FIX: Create phantom campaign sales records
# Phantom campaigns should have their own sales records in the PowerBI data
phantom_sales_records = []
if len(phantom_campaigns) > 0:
    for phantom_id in phantom_campaigns:
        # Check if phantom campaign has direct sales records
        phantom_direct_sales = df[df["cm_no"] == phantom_id]
        if not phantom_direct_sales.empty:
            phantom_sales_total = phantom_direct_sales["amount"].sum()
            phantom_buyers = phantom_direct_sales["customer_no"].nunique()
            
            # Create synthetic record for phantom campaign
            phantom_sales_records.append({
                "bundle_id": phantom_id,
                "amount": phantom_sales_total,
                "customer_no": phantom_direct_sales["customer_no"].iloc[0] if not phantom_direct_sales.empty else "",
                "cm_no": phantom_id,
                "main_cm": "",  # Phantom is main campaign itself
                "item_no": phantom_direct_sales["item_no"].iloc[0] if "item_no" in phantom_direct_sales.columns and not phantom_direct_sales.empty else "",
                "line_date": phantom_direct_sales["line_date"].iloc[0] if "line_date" in phantom_direct_sales.columns and not phantom_direct_sales.empty else pd.NaT
            })

# Add phantom records to main dataframe if they exist
if phantom_sales_records:
    phantom_df = pd.DataFrame(phantom_sales_records)
    
    # Ensure phantom campaigns are not double-counted by removing them from regular bundling
    df_enhanced = df.copy()
    
    # Add phantom records with proper bundle_id assignment
    for phantom_record in phantom_sales_records:
        phantom_record["bundle_id"] = phantom_record["cm_no"]  # Phantom campaigns bundle to themselves
    
    phantom_enhancement_df = pd.DataFrame(phantom_sales_records)
    df_enhanced = pd.concat([df_enhanced, phantom_enhancement_df], ignore_index=True)
else:
    df_enhanced = df.copy()

# Calculate sales by bundle (now includes phantom campaign direct sales)
sales_by_bundle = (df_enhanced.groupby("bundle_id")["amount"].sum()
                     .rename("sales_total").reset_index())

total_sales_all = sales_by_bundle["sales_total"].sum()

# Check phantom campaign contributions
phantom_sales_bundles = sales_by_bundle[sales_by_bundle["bundle_id"].isin(phantom_campaigns)]
phantom_sales_total = phantom_sales_bundles["sales_total"].sum() if len(phantom_sales_bundles) > 0 else 0

# Calculate buyers by bundle (includes phantom campaign attribution)
buyers_by_bundle = (df_enhanced.dropna(subset=["bundle_id","customer_no"])
                      .drop_duplicates(["bundle_id","customer_no"])
                      .groupby("bundle_id").size()
                      .rename("buyers_count").reset_index())

total_buyers_all = buyers_by_bundle["buyers_count"].sum()

bundles = (sales_by_bundle
           .merge(buyers_by_bundle, on="bundle_id", how="left")
           .merge(emails_by_bundle, on="bundle_id", how="left")
           .merge(inactive_count, on="bundle_id", how="left"))

for c in ["emails_total","buyers_count","inactive_recipients"]:
    if c in bundles.columns:
        bundles[c] = pd.to_numeric(bundles[c], errors="coerce").fillna(0).astype(int)

bundles["emails_total_effective"] = (bundles["emails_total"] - bundles["inactive_recipients"]).clip(lower=0).astype(int)
bundles["conversion_effective"] = np.where(bundles["emails_total_effective"]>0,
                                           bundles["buyers_count"]/bundles["emails_total_effective"], 0.0)

# === 6) Main item + wine name (using enhanced data) ===
item_sales = (df_enhanced.groupby(["bundle_id","item_no"])["amount"].sum()
                .rename("item_sales").reset_index())
top_item = (item_sales.sort_values(["bundle_id","item_sales"], ascending=[True, False])
                     .groupby("bundle_id").head(1)
                     .rename(columns={"item_no":"main_item_no","item_sales":"main_item_sales"}))

# attach wine name (use original df for wine mapping as phantom records may not have complete item info)
if name_col:
    tmp = df[[df.columns[i_item] if i_item is not None else "item_no", name_col]].dropna().copy()
    tmp.columns = ["item_no","wine_name"]
    tmp["item_no"] = tmp["item_no"].map(_norm_item)
    wine_map = (tmp.groupby(["item_no","wine_name"]).size()
                  .reset_index(name="n")
                  .sort_values(["item_no","n"], ascending=[True, False])
                  .drop_duplicates("item_no")[["item_no","wine_name"]])
    top_item["main_item_no"] = top_item["main_item_no"].map(_norm_item)
    top_item = top_item.merge(wine_map.rename(columns={"item_no":"main_item_no"}),
                              on="main_item_no", how="left").rename(columns={"wine_name":"main_wine_name"})

bundles = bundles.merge(top_item, on="bundle_id", how="left")

# === 7) Resurrection bonus (using enhanced data) ===
resurrect = pd.DataFrame(columns=["bundle_id","resurrection_count"])
if inactive_df is not None and not inactive_df.empty and "customer_no" in df_enhanced.columns and "line_date" in df_enhanced.columns:
    I_ct = _pick(inactive_df.columns, "contact no.", "customer no.", "contact id", regex=r"^(contact|customer).*(no|id)")
    if I_ct is not None:
        inactive_set = set(inactive_df.iloc[:, I_ct].astype(str).str.strip().str.replace(r"\.0$","", regex=True))
        df2025 = df_enhanced.loc[(df_enhanced["line_date"].notna()) & (df_enhanced["line_date"] >= pd.Timestamp("2025-01-01"))].copy()
        df2025["customer_no"] = df2025["customer_no"].astype(str).str.strip().str.replace(r"\.0$","", regex=True)
        df2025["is_inactive"] = df2025["customer_no"].isin(inactive_set)
        first = (df2025.sort_values("line_date")
                        .loc[df2025["is_inactive"]]
                        .drop_duplicates(["customer_no"])
                        .loc[:, ["customer_no","bundle_id"]])
        resurrect = (first.groupby("bundle_id")["customer_no"].nunique()
                           .rename("resurrection_count").reset_index())

bundles = bundles.merge(resurrect, on="bundle_id", how="left")
bundles["resurrection_count"] = pd.to_numeric(bundles["resurrection_count"], errors="coerce").fillna(0).astype(int)
bundles["resurrection_bonus"] = (RESURRECTION_BONUS_PER * bundles["resurrection_count"]).clip(upper=RESURRECTION_CAP)

# === 8) Final score ===
bundles["sales_norm"] = _minmax(bundles["sales_total"])
bundles["conv_norm"]  = _minmax(bundles["conversion_effective"])
bundles["weighted_core"] = (CONV_WEIGHT*bundles["conv_norm"] + SALES_WEIGHT*bundles["sales_norm"]).fillna(0.0)
bundles["success_score"] = (bundles["weighted_core"] + bundles["resurrection_bonus"]).fillna(0.0)

winners = (bundles.sort_values(["success_score","sales_total"], ascending=[False, False], kind="mergesort")
                  .reset_index(drop=True))
winners.insert(0, "rank", winners.index + 1)

# === 9) OMT schedule per bundle ===
def _pick_omt_sched(df):
    for c in df.columns:
        if re.search(r"(schedule).*(date|time)", str(c), re.I): return c
    for c in df.columns:
        if re.search(r"(send|scheduled)", str(c), re.I): return c
    return None

col_sched = _pick_omt_sched(omt_df)
omt_sched = omt_df[[omt_df.columns[j_cm], col_sched]].copy() if (col_sched and j_cm is not None) else pd.DataFrame(columns=["cm_no","schedule_dt"])
if not omt_sched.empty:
    omt_sched.columns = ["cm_no","schedule_dt"]
    omt_sched["cm_no"] = omt_sched["cm_no"].map(_norm_cm)
    omt_sched["schedule_dt"] = pd.to_datetime(omt_sched["schedule_dt"], errors="coerce")
    sched_by_bundle = (omt_sched.merge(cm_to_bundle, on="cm_no", how="left")
                                 .dropna(subset=["bundle_id"])
                                 .groupby("bundle_id", as_index=False)["schedule_dt"].min())
else:
    sched_by_bundle = pd.DataFrame(columns=["bundle_id","schedule_dt"])

# === 10) FINAL TABLE WITH ENHANCED VINTAGE EXTRACTION ===

# Apply 30-day filter
today = pd.Timestamp.today().normalize()
cutoff = today - pd.Timedelta(days=30)
sched_recent = sched_by_bundle[pd.to_datetime(sched_by_bundle["schedule_dt"], errors="coerce") >= cutoff]
w_recent = winners.merge(sched_recent, on="bundle_id", how="inner")

# Re-rank after filtering to start from 1
w_recent = w_recent.sort_values(["success_score","sales_total"], ascending=[False, False], kind="mergesort").reset_index(drop=True)
w_recent["rank"] = w_recent.index + 1

# Get top 25
top25 = w_recent.head(25).copy()

# CM formatting
def _fmt_cm(bid):
    s = str(bid).strip()
    if s and s.lower() != "nan":
        if len(s) >= 7:  # e.g., 2502233 -> CM-25-02233
            return f"CM-{s[:2]}-{s[2:]}"
        elif len(s) >= 5:  # e.g., 02233 -> CM-25-02233
            return f"CM-25-{s}"
        else:
            return f"CM-25-{s.zfill(5)}"
    return None

top25["Starting Date"] = pd.to_datetime(top25["schedule_dt"], errors="coerce").dt.date
top25["CM"] = top25["bundle_id"].apply(_fmt_cm)

# Recalculate metrics (using enhanced data with phantom campaigns)
sales_recalc = df_enhanced.groupby("bundle_id")["amount"].sum().rename("sales_total_fresh").reset_index()
buyers_recalc = (df_enhanced.dropna(subset=["bundle_id","customer_no"])
                .drop_duplicates(["bundle_id","customer_no"])
                .groupby("bundle_id").size()
                .rename("buyers_count_fresh").reset_index())
emails_recalc = emails_by_bundle.copy().rename(columns={"emails_total": "emails_total_fresh"})

top25 = (top25
         .merge(sales_recalc, on="bundle_id", how="left")
         .merge(buyers_recalc, on="bundle_id", how="left")
         .merge(emails_recalc, on="bundle_id", how="left"))

# Fill missing values
for col in ["sales_total_fresh", "buyers_count_fresh", "emails_total_fresh"]:
    if col in top25.columns:
        top25[col] = pd.to_numeric(top25[col], errors="coerce").fillna(0)

# ENHANCED VINTAGE EXTRACTION from multiple sources
def get_vintage_from_sources(bundle_id, wine_name):
    """Extract vintage from multiple data sources with priority order"""
    vintage = ""
    source = ""
    all_sources = []
    
    # Check all sources and collect them
    sources_found = {}
    
    # Source 1: OMT Vintage column
    if len(omt_vintage_map) > 0:
        omt_match = omt_vintage_map[omt_vintage_map["cm_no"] == bundle_id]
        if not omt_match.empty:
            omt_vintage = str(omt_match.iloc[0]["omt_vintage"]).strip()
            if omt_vintage and omt_vintage not in ["nan", "", "None", "0", "NaN"]:
                sources_found["OMT"] = omt_vintage
    
    # Source 2: Power BI Vintage column (use enhanced dataset)
    vintage_cols = [col for col in df_enhanced.columns if 'vintage' in col.lower()]
    if vintage_cols:
        vintage_col = vintage_cols[0]
        powerbi_match = df_enhanced[df_enhanced["bundle_id"] == bundle_id][vintage_col].dropna()
        if not powerbi_match.empty:
            # Get the most recent vintage if multiple entries
            powerbi_vintages = powerbi_match.unique()
            # Sort to get the newest vintage
            valid_vintages = []
            for v in powerbi_vintages:
                v_str = str(v).strip()
                if v_str and v_str not in ["", "nan", "None", "0", "NaN"]:
                    try:
                        v_int = int(float(v_str))
                        if 1900 <= v_int <= 2030:  # Valid vintage range
                            valid_vintages.append(v_int)
                    except:
                        pass
            
            if valid_vintages:
                # Use the most recent vintage
                latest_vintage = max(valid_vintages)
                sources_found["PowerBI"] = str(latest_vintage)
    
    # Source 3: Extract from wine name
    if pd.notna(wine_name) and wine_name != "":
        wine_str = str(wine_name).strip()
        vintage_patterns = [
            r'\b(20[0-9]{2})\b',  # 2000-2099 (prioritize recent years)
            r'\b(19[0-9]{2})\b',  # 1900-1999
            r"'([0-9]{2})\b",  # '95, '03 etc
            r'\b([0-9]{2})\s*$',  # ending with 2 digits
        ]
        
        for pattern in vintage_patterns:
            vintage_match = re.search(pattern, wine_str)
            if vintage_match:
                if "'" in pattern:  # Handle abbreviated years
                    year_short = vintage_match.group(1)
                    if int(year_short) <= 30:
                        extracted_vintage = f"20{year_short}"
                    else:
                        extracted_vintage = f"19{year_short}"
                else:
                    extracted_vintage = vintage_match.group(1)
                
                # Validate extracted vintage
                try:
                    v_int = int(extracted_vintage)
                    if 1900 <= v_int <= 2030:
                        sources_found["WineName"] = extracted_vintage
                        break
                except:
                    pass
    
    # Priority selection with smart logic
    # 1. If OMT has data and it's recent (2015+), prefer it
    # 2. If PowerBI has more recent data than OMT, prefer PowerBI
    # 3. If wine name has most recent data, prefer it
    # 4. Fall back to any available source
    
    if "OMT" in sources_found:
        try:
            omt_year = int(sources_found["OMT"])
            if omt_year >= 2015:  # Recent OMT data is reliable
                vintage = sources_found["OMT"]
                source = "OMT"
        except:
            pass
    
    # Check if PowerBI has more recent data
    if "PowerBI" in sources_found and not vintage:
        try:
            powerbi_year = int(sources_found["PowerBI"])
            if not vintage or (vintage and int(vintage) < powerbi_year and powerbi_year >= 2015):
                vintage = sources_found["PowerBI"]
                source = "PowerBI"
        except:
            pass
    
    # Check wine name extraction as backup or if it's more recent
    if "WineName" in sources_found and not vintage:
        vintage = sources_found["WineName"]
        source = "WineName"
    
    # If still no vintage, use any available source
    if not vintage and sources_found:
        for src, val in sources_found.items():
            vintage = val
            source = src
            break
    
    return vintage, source

# Apply enhanced vintage extraction
vintage_results = []
vintage_sources = []
wine_names_clean = []

for idx, row in top25.iterrows():
    bundle_id = row["bundle_id"]
    wine_name = row.get("main_wine_name", "")
    
    vintage, source = get_vintage_from_sources(bundle_id, wine_name)
    vintage_results.append(vintage)
    vintage_sources.append(source)
    
    # Clean wine name (remove vintage if extracted from name)
    if pd.notna(wine_name) and wine_name != "":
        wine_clean = str(wine_name).strip()
        if source == "WineName" and vintage:
            # Remove the vintage from wine name
            wine_clean = re.sub(rf'\b{vintage}\b', '', wine_clean).strip()
            wine_clean = re.sub(r'\s+', ' ', wine_clean).strip("- ")
        
        if len(wine_clean) > 30:
            wine_clean = wine_clean[:27] + "..."
    else:
        wine_clean = ""
    
    wine_names_clean.append(wine_clean)

top25["vintage"] = vintage_results
top25["vintage_source"] = vintage_sources  
top25["wine_name_short"] = wine_names_clean

# Create final display table
winners_top25_excel = pd.DataFrame({
    "CM":                top25["CM"],
    "Wine Name":         top25["wine_name_short"],
    "Vintage":           top25["vintage"],
    "Starting Date":     top25["Starting Date"],
    "Sales Total":       top25["sales_total_fresh"],
    "Clients":           top25["buyers_count_fresh"].astype("Int64"),
    "Total Sent":        top25["emails_total_fresh"].astype("Int64"),
    "Conversion":        pd.to_numeric(top25["conversion_effective"], errors="coerce").fillna(0.0).round(4),
    "Norm Sales":        (lambda s: ((s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s*0 + 1.0))
                         (pd.to_numeric(top25["sales_total_fresh"], errors="coerce").fillna(0.0)).round(4),
    "Norm Conv":         (lambda s: ((s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s*0 + 1.0))
                         (pd.to_numeric(top25["conversion_effective"], errors="coerce").fillna(0.0)).round(4),
    "Score":             (
        0.6 * (lambda s: ((s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s*0 + 1.0))
              (pd.to_numeric(top25["conversion_effective"], errors="coerce").fillna(0.0))
        + 0.4 * (lambda s: ((s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s*0 + 1.0))
              (pd.to_numeric(top25["sales_total_fresh"], errors="coerce").fillna(0.0))
        + (0.02 * pd.to_numeric(top25["resurrection_count"], errors="coerce").fillna(0)).clip(upper=0.10)
    ).round(6),
    "Rank":              pd.to_numeric(top25["rank"], errors="coerce").astype("Int64"),
    "Inactive":          pd.to_numeric(top25["resurrection_count"], errors="coerce").fillna(0).astype("Int64"),
})

# Sort by rank to ensure proper ordering
winners_top25_excel = winners_top25_excel.sort_values('Rank')

# Display the enhanced winners table with phantom campaign integration
print(f"🏆 TOP {len(winners_top25_excel)} WINE CAMPAIGN WINNERS (Last 30 Days)")
print(f"📧 Enhanced with Phantom Campaign Email Integration")
print("="*100)
print(winners_top25_excel.to_string(index=False))

# === ENHANCED DEBUGGING FOR CM-25-02493 AND CM-25-02472 ===
print(f"\n" + "="*100)
print(f"🔍 DETAILED INVESTIGATION: CM-25-02493 (child) & CM-25-02472 (parent)")
print(f"="*100)

target_campaigns = ["2502493", "2502472"]  # Bundle IDs without CM- prefix
target_cm_formatted = ["CM-25-02493", "CM-25-02472"]

for i, (bundle_id, cm_formatted) in enumerate(zip(target_campaigns, target_cm_formatted)):
    print(f"\n{'='*80}")
    print(f"📊 ANALYZING {cm_formatted} (Bundle ID: {bundle_id})")
    print(f"{'='*80}")
    
    # 1. Check if it exists in PowerBI data (original df)
    powerbi_records = df[df["cm_no"] == bundle_id]
    powerbi_main_records = df[df["main_cm"] == bundle_id]
    
    print(f"\n📊 PowerBI Data Analysis:")
    print(f"   • Direct records (cm_no = {bundle_id}): {len(powerbi_records)}")
    print(f"   • Referenced as main campaign: {len(powerbi_main_records)}")
    
    if len(powerbi_records) > 0:
        total_sales = powerbi_records["amount"].sum()
        unique_customers = powerbi_records["customer_no"].nunique()
        print(f"   • Direct sales: {total_sales:,.2f}")
        print(f"   • Direct customers: {unique_customers}")
        
        # Show sample records with more detail
        print(f"   • Sample records:")
        for idx, row in powerbi_records.head(3).iterrows():
            wine_info = f", Wine: {row.get(name_col, 'N/A')[:30]}" if name_col and pd.notna(row.get(name_col)) else ""
            print(f"     - Amount: {row['amount']:,.2f}, Customer: {row['customer_no']}, Main CM: '{row['main_cm']}'{wine_info}")
    else:
        print(f"   • No direct sales records found")
    
    if len(powerbi_main_records) > 0:
        main_sales = powerbi_main_records["amount"].sum()
        main_customers = powerbi_main_records["customer_no"].nunique()
        print(f"   • Sales from sub-campaigns: {main_sales:,.2f}")
        print(f"   • Customers from sub-campaigns: {main_customers}")
        
        # Show which campaigns reference this as main
        sub_campaigns = powerbi_main_records["cm_no"].unique()
        print(f"   • Sub-campaigns ({len(sub_campaigns)} total): {sub_campaigns[:5]}{'...' if len(sub_campaigns) > 5 else ''}")
        
        # Show sample sub-campaign records
        print(f"   • Sample sub-campaign records:")
        for idx, row in powerbi_main_records.head(3).iterrows():
            wine_info = f", Wine: {row.get(name_col, 'N/A')[:30]}" if name_col and pd.notna(row.get(name_col)) else ""
            print(f"     - Sub CM: {row['cm_no']}, Amount: {row['amount']:,.2f}{wine_info}")
    else:
        print(f"   • No sub-campaigns reference this as main")
    
    # 2. Check if it's a phantom campaign
    is_phantom = bundle_id in phantom_campaigns
    print(f"\n👻 Phantom Campaign Status:")
    print(f"   • Is phantom: {'✅ Yes' if is_phantom else '❌ No'}")
    if is_phantom:
        print(f"   • Phantom campaigns detected: {len(phantom_campaigns)}")
    
    # 3. Check OMT email data
    omt_records = omt_unique[omt_unique["cm_no"] == bundle_id]
    print(f"\n📧 OMT Email Data:")
    print(f"   • OMT records found: {len(omt_records)}")
    
    if len(omt_records) > 0:
        total_emails = omt_records["emails_sent"].sum()
        print(f"   • Total emails sent: {total_emails:,}")
        
        # Show OMT record details
        for idx, row in omt_records.iterrows():
            size_info = f", Size: {row.get('size', 'N/A')}" if 'size' in row else ""
            minqty_info = f", MinQty: {row.get('min_qty', 'N/A')}" if 'min_qty' in row else ""
            print(f"     - Emails: {row['emails_sent']:,}{size_info}{minqty_info}")
    else:
        print(f"   • No OMT email records found for {bundle_id}")
        
        # Check if there are any similar campaign numbers
        similar_campaigns = omt_unique[omt_unique["cm_no"].str.contains(bundle_id[-5:], na=False)]
        if len(similar_campaigns) > 0:
            print(f"   • Similar campaign numbers in OMT:")
            for idx, row in similar_campaigns.head(3).iterrows():
                print(f"     - CM: {row['cm_no']}, Emails: {row['emails_sent']:,}")
    
    # 4. Check bundle aggregation
    bundle_check = bundles[bundles["bundle_id"] == bundle_id]
    print(f"\n🎯 Bundle Aggregation:")
    print(f"   • Found in bundles table: {'✅ Yes' if len(bundle_check) > 0 else '❌ No'}")
    
    if len(bundle_check) > 0:
        bundle_row = bundle_check.iloc[0]
        print(f"   • Sales total: {bundle_row['sales_total']:,.2f}")
        print(f"   • Buyers count: {bundle_row['buyers_count']}")
        print(f"   • Emails total: {bundle_row['emails_total']}")
        print(f"   • Emails effective: {bundle_row['emails_total_effective']}")
        print(f"   • Conversion effective: {bundle_row['conversion_effective']:.4f}")
        print(f"   • Success score: {bundle_row['success_score']:.6f}")
        
        # Check if it has wine name
        if 'main_wine_name' in bundle_row and pd.notna(bundle_row['main_wine_name']):
            print(f"   • Wine name: {bundle_row['main_wine_name']}")
    
    # 5. Check OMT schedule data
    sched_check = omt_sched[omt_sched["cm_no"] == bundle_id] if not omt_sched.empty else pd.DataFrame()
    print(f"\n📅 Schedule Data:")
    print(f"   • Schedule records found: {len(sched_check)}")
    
    if len(sched_check) > 0:
        schedule_date = sched_check.iloc[0]["schedule_dt"]
        print(f"   • Schedule date: {schedule_date}")
        if pd.notna(schedule_date):
            is_within_30_days = schedule_date >= cutoff
            days_diff = (today - schedule_date.normalize()).days
            print(f"   • Within 30 days: {'✅ Yes' if is_within_30_days else '❌ No'}")
            print(f"   • Days ago: {days_diff}")
            print(f"   • Cutoff date: {cutoff}")
    else:
        print(f"   • No schedule data found")
        
        # Check for similar schedule records
        similar_sched = omt_sched[omt_sched["cm_no"].str.contains(bundle_id[-5:], na=False)] if not omt_sched.empty else pd.DataFrame()
        if len(similar_sched) > 0:
            print(f"   • Similar campaigns in schedule:")
            for idx, row in similar_sched.head(3).iterrows():
                print(f"     - CM: {row['cm_no']}, Date: {row['schedule_dt']}")
    
    # 6. Check if it appears in winners (any rank)
    winner_check = winners[winners["bundle_id"] == bundle_id]
    print(f"\n🏆 Winners Analysis:")
    print(f"   • Found in full winners list: {'✅ Yes' if len(winner_check) > 0 else '❌ No'}")
    
    if len(winner_check) > 0:
        winner_row = winner_check.iloc[0]
        print(f"   • Overall rank: {winner_row['rank']}")
        print(f"   • Success score: {winner_row['success_score']:.6f}")
        print(f"   • Sales total: {winner_row['sales_total']:,.2f}")
        print(f"   • In TOP 25: {'✅ Yes' if winner_row['rank'] <= 25 else '❌ No'}")
    
    # 7. Check recent winners (30-day filtered)
    recent_check = w_recent[w_recent["bundle_id"] == bundle_id]
    print(f"\n📊 Recent Winners (30-day filtered):")
    print(f"   • Found in recent winners: {'✅ Yes' if len(recent_check) > 0 else '❌ No'}")
    
    if len(recent_check) > 0:
        recent_row = recent_check.iloc[0]
        print(f"   • Recent rank: {recent_row['rank']}")
        print(f"   • Success score: {recent_row['success_score']:.6f}")
        print(f"   • In recent TOP 25: {'✅ Yes' if recent_row['rank'] <= 25 else '❌ No'}")
    
    # 8. Final check in TOP 25 table
    final_check = winners_top25_excel[winners_top25_excel["CM"] == cm_formatted]
    print(f"\n🎯 Final TOP 25 Table:")
    print(f"   • Found in TOP 25 table: {'✅ Yes' if len(final_check) > 0 else '❌ No'}")
    
    if len(final_check) > 0:
        final_row = final_check.iloc[0]
        print(f"   • Final rank: {final_row['Rank']}")
        print(f"   • Final score: {final_row['Score']:.6f}")
        print(f"   • Sales total: {final_row['Sales Total']:,.2f}")
        print(f"   • Emails total: {final_row['Total Sent']:,}")

# Summary of issues found
print(f"\n" + "="*100)
print(f"📋 INVESTIGATION SUMMARY")
print(f"="*100)
print(f"• Total phantom campaigns detected: {len(phantom_campaigns)}")
print(f"• Total bundles in analysis: {len(bundles)}")
print(f"• Total winners (all ranks): {len(winners)}")
print(f"• Winners with recent schedule (30-day): {len(w_recent)}")
print(f"• Final TOP 25 count: {len(winners_top25_excel)}")
print(f"• Current 30-day cutoff: {cutoff}")
print(f"• Analysis date: {today}")

# Analysis completed - clean output with only the main winners table above


🔍 OMT COLUMN DEBUGGING:
   • Total columns: 22
   • All columns: ['campaign status', 'schedule datetime', 'number of sent emails', 'campaign no.', 'campaign description', 'item no.', 'item description', 'minimum quantity', 'unit price', 'unit price (eur)', 'size', 'size1', 'wine name', 'wine no.', 'producer name', 'region code', 'sub-region code', 'vintage', 'color code', 'producer name1', 'competitor code', 'schedule_datetime']
   • Campaign column candidates: [(0, 'campaign status'), (3, 'campaign no.'), (4, 'campaign description'), (5, 'item no.'), (13, 'wine no.')]
   • Email column candidates: [(2, 'number of sent emails')]
   • Selected indices - Campaign: 3, Email: 2, Size: 10, MinQty: 7, Vintage: 17
   • Campaign column: 'campaign no.'
   • Email column: 'number of sent emails'

🔍 SEARCHING FOR TARGET CAMPAIGNS IN OMT:
   • Searching for '2502493': 2 matches
     - CM: '2502493', Emails: 44
     - CM: '2502493', Emails: 44
   • Searching for '2502472': 3 matches
     - CM: '25

In [8]:
# 🏆 FINAL RESULTS: TOP 25 WINE CAMPAIGN WINNERS (Enhanced with Phantom Campaign Logic)
print("🏆 TOP 25 WINE CAMPAIGN WINNERS")
print("=" * 50)
print(f"📊 Analysis Date: {today.strftime('%B %d, %Y')}")
print(f"📈 Total Campaigns Analyzed: {len(bundles):,}")
print(f"👻 Phantom Campaigns Included: {len(phantom_campaigns)}")
print(f"💰 Total Sales: ${total_sales_all:,.2f}")
print(f"📧 Total Emails: {total_emails_all:,}")
print(f"👥 Total Buyers: {total_buyers_all:,}")
print("=" * 50)
print()

# Display the clean winners table
display(top25)

🏆 TOP 25 WINE CAMPAIGN WINNERS
📊 Analysis Date: September 30, 2025
📈 Total Campaigns Analyzed: 1,441
👻 Phantom Campaigns Included: 8
💰 Total Sales: $52,461,388.78
📧 Total Emails: 1,288,581
👥 Total Buyers: 16,473



,rank,bundle_id,sales_total,buyers_count,emails_total,inactive_recipients,emails_total_effective,conversion_effective,main_item_no,main_item_sales,...,success_score,schedule_dt,Starting Date,CM,sales_total_fresh,buyers_count_fresh,emails_total_fresh,vintage,vintage_source,wine_name_short
0,1,2502259,200833.606698,11,228,0,228,0.048246,65449,58065.880000,...,0.225498,2025-09-02 12:13:15.497,2025-09-02,CM-25-02259,200833.606698,11,228,2023,OMT,Masseto
1,2,2502341,35755.113190,110,2548,0,2548,0.043171,35837,35755.113190,...,0.199267,2025-09-08 16:25:00.000,2025-09-08,CM-25-02341,35755.113190,110,2548,2015,OMT,Retout
2,3,2502383,18658.282383,94,2371,0,2371,0.039646,35837,14481.446120,...,0.182751,2025-09-11 11:02:00.000,2025-09-11,CM-25-02383,18658.282383,94,2371,2019,PowerBI,Retout
3,4,2502451,211010.522937,104,2746,0,2746,0.037873,48976,182826.980652,...,0.177963,2025-09-16 16:04:00.000,2025-09-16,CM-25-02451,211010.522937,104,2746,2018,OMT,Figeac
4,5,2501623,11750.146923,19,493,0,493,0.038540,65245,6519.224603,...,0.177542,2025-09-09 11:01:00.000,2025-09-09,CM-25-01623,11750.146923,19,493,2023,PowerBI,Oreno
5,6,2502583,76565.624422,5,152,0,152,0.032895,62292,67179.685476,...,0.152710,2025-09-26 16:32:00.000,2025-09-26,CM-25-02583,76565.624422,5,152,2019,OMT,Cabernet Sauvignon
6,7,2502427,47460.959086,61,1907,0,1907,0.031987,65509,47460.959086,...,0.148027,2025-09-15 10:30:50.740,2025-09-15,CM-25-02427,47460.959086,61,1907,2021,OMT,Yjar
7,8,2502586,52258.058770,23,770,0,770,0.029870,65703,28106.955578,...,0.138371,2025-09-26 10:35:48.277,2025-09-26,CM-25-02586,52258.058770,23,770,2015,PowerBI,Champagne Dom Pérignon Spec...
8,9,2502276,29796.171700,33,1377,0,1377,0.023965,64179,23868.967343,...,0.130815,2025-09-03 10:15:00.000,2025-09-03,CM-25-02276,29796.171700,33,1377,2023,OMT,Almaviva
9,10,2502394,2347.920000,3,106,0,106,0.028302,35837,2347.920000,...,0.130284,2025-09-11 14:11:17.537,2025-09-11,CM-25-02394,2347.920000,3,106,2015,OMT,Retout
